# Phase 8: Final Report -- Tables, Figures, and Documentation

This phase compiles the project's results into paper-ready artifacts:

1. **Tables** (LaTeX `.tex` + 300dpi PNG, saved to `results/tables/`):
   dataset statistics, per-class metrics with bootstrap 95% CIs, a comparison
   to published benchmarks, and an ablation study of the training/decision
   pipeline.
2. **Figures** (300dpi PNG, saved to `results/figures/`, captions provided in
   markdown rather than in-image titles): class distribution + samples,
   training curves, confusion matrix, ROC curves, Grad-CAM++ examples, and a
   mean Grad-CAM++ heatmap per class.
3. **Limitations**, **ethical statement**, and a **reproducibility package**
   (`reproduce.md`).

This notebook only *reads* existing artifacts from Phases 1-7 (metrics CSVs,
the trained model, the Grad-CAM++ implementation) -- no retraining is
performed.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cv2
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision.models import efficientnet_b4

from config_loader import load_config, resolve_image_path

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BASE_DIR    = Path(r'c:\graduation project')
DATA_DIR    = BASE_DIR / 'data'
MODELS_DIR  = BASE_DIR / 'models' / 'checkpoints'
METRICS_DIR = BASE_DIR / 'results' / 'metrics'
FIGURES_DIR = BASE_DIR / 'results' / 'figures'
TABLES_DIR  = BASE_DIR / 'results' / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

cfg = load_config(BASE_DIR)
CLASSES     = cfg['classes']
NUM_CLASSES = cfg['num_classes']
NORM_MEAN   = cfg['norm_mean']
NORM_STD    = cfg['norm_std']
IMG_SIZE    = cfg['img_size']
MALIGNANT_CLASSES = ['melanoma', 'basal cell carcinoma', 'squamous cell carcinoma']

DPI = 300


def class_label(c):
    return c.replace('_', ' ').title()


print(f"Device: {DEVICE}")
print(f"Classes ({NUM_CLASSES}): {CLASSES}")

In [ ]:
def save_table(df: pd.DataFrame, name: str, caption: str, col_format: str = None,
                float_format: str = '%.3f'):
    """Save a DataFrame as LaTeX (.tex) and a 300dpi PNG render."""
    tex_path = TABLES_DIR / f'{name}.tex'
    png_path = TABLES_DIR / f'{name}.png'

    latex = df.to_latex(index=False, caption=caption, label=f'tab:{name}',
                         column_format=col_format, float_format=float_format,
                         escape=True)
    tex_path.write_text(latex, encoding='utf-8')

    fig_h = 0.5 + 0.35 * len(df)
    fig_w = max(8, 1.4 * len(df.columns))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis('off')
    cell_text = df.astype(str).values
    table = ax.table(cellText=cell_text, colLabels=df.columns, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.4)
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight='bold')
            cell.set_facecolor('#dde6f2')
    fig.tight_layout()
    fig.savefig(png_path, dpi=DPI, bbox_inches='tight')
    plt.close(fig)

    print(f"Saved {tex_path.name} and {png_path.name} ({len(df)} rows)")


print("save_table() defined ✓")

## Table 1 -- Dataset Statistics

Per-class image counts across the train/validation/test splits, source
dataset composition (HAM10000 vs. ISIC), and the malignancy flag used to
define `MALIGNANT_CLASSES`.

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'val.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')
eda_df   = pd.read_csv(METRICS_DIR / 'eda_summary.csv')

rows = []
for _, r in eda_df.iterrows():
    cls = r['label_str']
    n_train = (train_df['label_str'] == cls).sum()
    n_val   = (val_df['label_str'] == cls).sum()
    n_test  = (test_df['label_str'] == cls).sum()
    n_total = n_train + n_val + n_test
    rows.append({
        'Class': class_label(cls),
        'Train': n_train,
        'Val': n_val,
        'Test': n_test,
        'Total': n_total,
        '% of Dataset': round(100 * n_total / len(eda_df['count']) if False else r['pct'], 2),
        'Malignant': 'Yes' if r['is_malignant'] else 'No',
    })

table1 = pd.DataFrame(rows)
total_row = {
    'Class': 'Total',
    'Train': table1['Train'].sum(),
    'Val': table1['Val'].sum(),
    'Test': table1['Test'].sum(),
    'Total': table1['Total'].sum(),
    '% of Dataset': 100.0,
    'Malignant': '-',
}
table1 = pd.concat([table1, pd.DataFrame([total_row])], ignore_index=True)

print(table1.to_string(index=False))
print()
print("Source datasets (test split):", test_df['source_dataset'].value_counts().to_dict())
print("Source datasets (train split):", train_df['source_dataset'].value_counts().to_dict())
print("Source datasets (val split):", val_df['source_dataset'].value_counts().to_dict())

save_table(table1, 'table1_dataset_statistics',
           caption='Per-class image counts across train/validation/test splits.',
           col_format='lrrrrrl', float_format='%.2f')

## Table 2 -- Per-Class Test Metrics with Bootstrap 95% CIs

Per-class sensitivity (recall) and AUC are reported with 95% confidence
intervals from 1000 bootstrap resamples of the test set (n=1715, `SEED=42`),
using the **argmax** decision rule. Specificity, precision, and F1 are point
estimates from `test_per_class_metrics.csv` (Phase 5).

In [ ]:
test_pred_df = pd.read_csv(METRICS_DIR / 'test_predictions.csv')
per_class_df = pd.read_csv(METRICS_DIR / 'test_per_class_metrics.csv')
proba_cols = [f"proba_{c.replace(' ', '_')}" for c in CLASSES]

y_true = test_pred_df['label_int'].to_numpy()
y_pred = test_pred_df['pred_int'].to_numpy()
proba  = test_pred_df[proba_cols].to_numpy()
n = len(test_pred_df)

N_BOOT = 1000
boot_sens = {c: [] for c in range(NUM_CLASSES)}
boot_auc  = {c: [] for c in range(NUM_CLASSES)}

for _ in range(N_BOOT):
    idx = rng.integers(0, n, n)
    yt, yp, pr = y_true[idx], y_pred[idx], proba[idx]
    for c in range(NUM_CLASSES):
        binary_true = (yt == c).astype(int)
        if 0 < binary_true.sum() < n:
            tp = int(((yp == c) & (yt == c)).sum())
            fn = int(((yp != c) & (yt == c)).sum())
            boot_sens[c].append(tp / (tp + fn) if (tp + fn) > 0 else np.nan)
            boot_auc[c].append(roc_auc_score(binary_true, pr[:, c]))

rows = []
for i, cls in enumerate(CLASSES):
    pc = per_class_df[per_class_df['class'] == cls].iloc[0]
    sens_lo, sens_hi = np.percentile(boot_sens[i], [2.5, 97.5])
    auc_lo, auc_hi = np.percentile(boot_auc[i], [2.5, 97.5])
    rows.append({
        'Class': class_label(cls),
        'Malignant': 'Yes' if pc['malignant'] else 'No',
        'Support': int(pc['support']),
        'Sensitivity [95% CI]': f"{pc['sensitivity_recall']:.3f} [{sens_lo:.3f}, {sens_hi:.3f}]",
        'Specificity': f"{pc['specificity']:.3f}",
        'Precision': f"{pc['precision']:.3f}",
        'F1': f"{pc['f1']:.3f}",
        'AUC [95% CI]': f"{pc['auc']:.3f} [{auc_lo:.3f}, {auc_hi:.3f}]",
    })

table2 = pd.DataFrame(rows)
print(table2.to_string(index=False))

save_table(table2, 'table2_per_class_metrics',
           caption='Per-class test set metrics (argmax decision rule) with bootstrap 95\\% confidence intervals (1000 resamples).',
           col_format='lllllll l')

## Table 3 -- Comparison to Published Benchmarks

This project's 8-class label set (7 HAM10000 classes + squamous cell
carcinoma from ISIC) matches the **ISIC 2019** challenge task. The closest
published reference is the ISIC 2019 challenge-winning ensemble (Gessert et
al., 2020), evaluated on the official held-out ISIC 2019 test set. An
EfficientNet-B4 result on the *standard 7-class HAM10000* task (different
label set and split) is included for additional architectural context.

**Caveat (also repeated in Limitations)**: these numbers are **not directly
comparable** -- different test splits, different class counts (7 vs. 8),
different primary metrics, and (for the HAM10000-only studies) a higher risk
of image-level train/test leakage from near-duplicate lesion photos. They are
included to situate this project's results relative to the literature, not as
a claim of outperforming or underperforming any specific prior work.

In [ ]:
macro_f1_argmax = per_class_df['f1'].mean()
macro_auc = pd.read_csv(METRICS_DIR / 'test_bootstrap_ci.csv')
macro_auc_val = macro_auc.loc[macro_auc['metric'] == 'macro_auc', 'point_estimate'].iloc[0]
acc_argmax = (y_true == y_pred).mean()

table3 = pd.DataFrame([
    {
        'Study': 'Gessert et al. (2020) -- ISIC 2019 winner',
        'Architecture': 'EfficientNet ensemble (B0-B6) + metadata',
        'Task / Test set': 'ISIC 2019 official test (8-class, n=8238)',
        'Accuracy': '0.748',
        'Macro F1': '0.743',
        'Macro AUC': 'n/r',
    },
    {
        'Study': 'Khalid et al. (2026), ETASR -- EfficientNet-B4 + Soft Attention',
        'Architecture': 'EfficientNet-B4 + soft attention',
        'Task / Test set': 'HAM10000 7-class (random split)',
        'Accuracy': '0.931',
        'Macro F1': '0.835',
        'Macro AUC': '0.990',
    },
    {
        'Study': 'This work',
        'Architecture': 'EfficientNet-B4 + SE block',
        'Task / Test set': 'HAM10000+ISIC 8-class, held-out test (n=1715)',
        'Accuracy': f'{acc_argmax:.3f}',
        'Macro F1': f'{macro_f1_argmax:.3f}',
        'Macro AUC': f'{macro_auc_val:.3f}',
    },
])

print(table3.to_string(index=False))

save_table(table3, 'table3_sota_comparison',
           caption='Comparison to published EfficientNet-based skin lesion classifiers (not directly comparable -- see text).',
           col_format='p{4.5cm}p{3.5cm}p{4.5cm}rrr')

## Table 4 -- Ablation Study

**(a) Progressive fine-tuning** -- the 3-stage training schedule from Phase 4
(`model_architecture.json`, `training_summary.csv`): each stage unfreezes more
of the EfficientNet-B4 backbone.

**(b) Decision rule** -- argmax vs. per-class Youden's-J-optimal thresholds
(Phase 5 threshold tuning, `test_target_comparison.csv`). Macro AUC is
threshold-independent and shown once for reference.

In [ ]:
training_summary = pd.read_csv(METRICS_DIR / 'training_summary.csv')
arch = json.loads((METRICS_DIR / 'model_architecture.json').read_text())

table4a = pd.DataFrame([
    {
        'Stage': int(s['stage']),
        'Description': arch['stages'][str(int(s['stage']))]['desc'],
        'Epochs': int(s['epochs_run']),
        'LR': arch['stages'][str(int(s['stage']))]['lr'],
        'Best Val Macro AUC': f"{s['best_val_auc']:.4f}",
    }
    for _, s in training_summary.iterrows()
])
print(table4a.to_string(index=False))
save_table(table4a, 'table4a_progressive_finetuning',
           caption='Ablation (a): effect of progressively unfreezing the EfficientNet-B4 backbone.',
           col_format='clrrr')

target_df = pd.read_csv(METRICS_DIR / 'test_target_comparison.csv')


def get_metric(name):
    return target_df.loc[target_df['metric'] == name, 'achieved'].iloc[0]


kappa_argmax = get_metric("Cohen's kappa (argmax)")
kappa_thresh = get_metric("Cohen's kappa (threshold-adj.)")

table4b = pd.DataFrame([
    {
        'Decision Rule': 'Argmax (default)',
        'Macro AUC': f"{macro_auc_val:.3f}",
        'Melanoma Sensitivity': f"{get_metric('Melanoma sensitivity (argmax)'):.3f}",
        "Cohen's Kappa": f"{kappa_argmax:.3f}",
        'Overall Accuracy': f"{get_metric('Overall accuracy (argmax)'):.3f}",
    },
    {
        'Decision Rule': "Per-class Youden's J threshold",
        'Macro AUC': f"{macro_auc_val:.3f}",
        'Melanoma Sensitivity': f"{get_metric('Melanoma sensitivity (threshold-adj.)'):.3f}",
        "Cohen's Kappa": f"{kappa_thresh:.3f}",
        'Overall Accuracy': f"{get_metric('Overall accuracy (threshold-adj.)'):.3f}",
    },
])
print()
print(table4b.to_string(index=False))
save_table(table4b, 'table4b_decision_rule',
           caption='Ablation (b): argmax vs. per-class Youden-optimal threshold decision rule.',
           col_format='lrrrr')


## Figure Suite (300dpi)

Six figures, each saved as a 300dpi PNG to `results/figures/`. Captions are
given here in markdown rather than as in-image titles, per the report
formatting requirements:

1. Class distribution and representative samples
2. Training curves across the 3-stage fine-tuning schedule
3. Confusion matrix (counts and row-normalized)
4. ROC curves for all 8 classes
5. Grad-CAM++ examples (correct vs. misclassified malignant cases)
6. Mean Grad-CAM++ heatmap per class

**Figure 1.** Class distribution across the combined dataset (top; bar
labels show count and percentage, malignant classes in red) and one
representative dermoscopy image per class (bottom).

In [ ]:
import matplotlib.patches as mpatches

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.1], hspace=0.4)

ax = fig.add_subplot(gs[0])
colors = ['#E24B4A' if c in MALIGNANT_CLASSES else '#378ADD' for c in eda_df['label_str']]
labels = [class_label(c) for c in eda_df['label_str']]
bars = ax.bar(labels, eda_df['count'], color=colors, edgecolor='white')
for bar, row in zip(bars, eda_df.itertuples()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
            f'{row.count:,}\n({row.pct}%)', ha='center', va='bottom', fontsize=8)
ax.set_ylabel('Image Count')
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.grid(axis='y', alpha=0.3)
ax.legend(handles=[
    mpatches.Patch(color='#E24B4A', label='Malignant'),
    mpatches.Patch(color='#378ADD', label='Benign / Pre-cancerous'),
])

gs_bottom = gs[1].subgridspec(1, NUM_CLASSES)
sample_rows = train_df.groupby('label_str', group_keys=False).apply(
    lambda g: g.sample(1, random_state=SEED), include_groups=False
)
sample_rows = sample_rows.join(train_df['label_str'])

for i, cls in enumerate(CLASSES):
    ax2 = fig.add_subplot(gs_bottom[i])
    row = sample_rows[sample_rows['label_str'] == cls].iloc[0]
    img = Image.open(row['image_path']).convert('RGB')
    ax2.imshow(img)
    ax2.axis('off')
    ax2.set_title(class_label(cls), fontsize=8)

fig.savefig(FIGURES_DIR / 'figure1_class_distribution_samples.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved figure1_class_distribution_samples.png")

**Figure 2.** Training and validation loss, validation macro AUC, and
validation melanoma sensitivity across the 3-stage progressive fine-tuning
schedule (dotted vertical lines mark stage boundaries; dashed gray lines mark
the project's target thresholds of 0.93 macro AUC and 0.85 melanoma
sensitivity).

In [ ]:
import matplotlib.lines as mlines

stage_dfs = {s: pd.read_csv(METRICS_DIR / f'history_stage{s}.csv') for s in [1, 2, 3]}
offsets = {}
cum = 0
for s in [1, 2, 3]:
    offsets[s] = cum
    cum += len(stage_dfs[s])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
stage_colors = {1: '#378ADD', 2: '#1D9E75', 3: '#E24B4A'}

for s in [1, 2, 3]:
    df_s = stage_dfs[s]
    x = df_s['epoch'] + offsets[s]
    axes[0].plot(x, df_s['train_loss'], color=stage_colors[s], marker='o', ms=3, linestyle='-')
    axes[0].plot(x, df_s['val_loss'], color=stage_colors[s], marker='s', ms=3, linestyle='--')
    axes[1].plot(x, df_s['val_auc_macro'], color=stage_colors[s], marker='o', ms=3)
    axes[2].plot(x, df_s['sens_melanoma'], color=stage_colors[s], marker='o', ms=3)

for ax in axes:
    for s in [1, 2]:
        ax.axvline(offsets[s + 1] - 0.5, color='gray', linestyle=':', alpha=0.6)
    ax.set_xlabel('Cumulative Epoch')
    ax.grid(alpha=0.3)

axes[0].set_title('Loss')
axes[0].legend(handles=[
    mlines.Line2D([], [], color='black', linestyle='-', marker='o', ms=4, label='Train'),
    mlines.Line2D([], [], color='black', linestyle='--', marker='s', ms=4, label='Validation'),
], fontsize=8)

axes[1].axhline(0.93, color='gray', linestyle=':', alpha=0.8)
axes[1].set_title('Validation Macro AUC')

axes[2].axhline(0.85, color='gray', linestyle=':', alpha=0.8)
axes[2].set_title('Validation Melanoma Sensitivity')

stage_handles = [mlines.Line2D([], [], color=stage_colors[s], marker='o', ms=5, label=f'Stage {s}')
                  for s in [1, 2, 3]]
fig.legend(handles=stage_handles, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.06), fontsize=9)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'figure2_training_curves.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved figure2_training_curves.png")

**Figure 3.** Test-set confusion matrix (argmax decision rule, n=1715):
raw counts (left) and row-normalized (right, values sum to 1 across each true
class).

In [ ]:
import seaborn as sns

cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
class_labels = [class_label(c) for c in CLASSES]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels, ax=axes[0])
axes[0].set_title('Counts')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels, ax=axes[1])
axes[1].set_title('Row-normalized')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

for ax in axes:
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    plt.setp(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'figure3_confusion_matrix.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved figure3_confusion_matrix.png")

**Figure 4.** Test-set ROC curves for all 8 classes (one-vs-rest), solid
lines for malignant classes and dashed lines for benign/pre-cancerous classes.
The diagonal dotted line marks chance-level performance (AUC=0.5).

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import auc as sk_auc

y_true_bin = label_binarize(y_true, classes=range(NUM_CLASSES))

fig, ax = plt.subplots(figsize=(8, 8))
auc_per_class = {}
for i, cls in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], proba[:, i])
    roc_auc = sk_auc(fpr, tpr)
    auc_per_class[cls] = roc_auc
    style = '-' if cls in MALIGNANT_CLASSES else '--'
    ax.plot(fpr, tpr, style, label=f'{class_label(cls)} (AUC={roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k:', alpha=0.4)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=8, loc='lower right')
ax.grid(alpha=0.3)

fig.savefig(FIGURES_DIR / 'figure4_roc_curves.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved figure4_roc_curves.png")
print(f"Macro AUC: {macro_auc_val:.4f}")

## Grad-CAM++ Setup (for Figures 5 & 6)

Reloads the final model and the Grad-CAM++ implementation from Phase 6
(`model.features[-1]`, the final 1792-channel/7x7 EfficientNet-B4 feature map,
is the target layer).

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class SkinCancerModel(nn.Module):
    IN_FEATURES = 1792

    def __init__(self, num_classes=8):
        super().__init__()
        base = efficientnet_b4(weights=None)
        self.features = base.features
        self.se = SEBlock(self.IN_FEATURES, reduction=16)
        self.avgpool = base.avgpool
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(self.IN_FEATURES),
            nn.Linear(self.IN_FEATURES, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.se(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


model = SkinCancerModel(num_classes=NUM_CLASSES).to(DEVICE)
state_dict = torch.load(MODELS_DIR / 'final_model.pth', map_location=DEVICE, weights_only=True)
model.load_state_dict(state_dict)
model.eval()


class GradCAMPlusPlus:
    ''' Grad-CAM++ via forward/backward hooks on a target conv layer. '''

    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor, class_idx, percentile_clip=90):
        self.model.zero_grad()
        output = self.model(input_tensor)
        score = output[:, class_idx]
        score.backward(retain_graph=True)

        grads = self.gradients
        acts = self.activations

        grads2 = grads ** 2
        grads3 = grads2 * grads
        sum_acts = acts.sum(dim=(2, 3), keepdim=True)
        eps = 1e-8
        alpha = grads2 / (2 * grads2 + sum_acts * grads3 + eps)
        alpha = torch.where(grads != 0, alpha, torch.zeros_like(alpha))
        weights = (alpha * F.relu(grads)).sum(dim=(2, 3), keepdim=True)

        cam = (weights * acts).sum(dim=1, keepdim=True)
        cam = F.relu(cam).squeeze().cpu().numpy()

        cap = np.percentile(cam, percentile_clip)
        if cap > 0:
            cam = np.minimum(cam, cap)

        cam_t = torch.from_numpy(cam).float().unsqueeze(0).unsqueeze(0)
        cam_t = F.interpolate(cam_t, size=input_tensor.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam_t.squeeze().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + eps)
        return cam, output.softmax(dim=1).detach().cpu().numpy()[0]


inference_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
])


def load_image_tensor(path):
    image = Image.open(path).convert('RGB')
    image = image.resize((IMG_SIZE, IMG_SIZE))
    tensor = inference_transform(image).unsqueeze(0).to(DEVICE)
    return image, tensor


def overlay_heatmap(image_pil, cam, alpha=0.45):
    image_np = np.asarray(image_pil).astype(np.float32) / 255.0
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    overlay = alpha * heatmap + (1 - alpha) * image_np
    return np.clip(overlay, 0, 1)


gradcam = GradCAMPlusPlus(model, model.features[-1])
print("Model + Grad-CAM++ ready")

**Figure 5.** Grad-CAM++ heatmaps for representative test cases (model
attention shown for the *predicted* class). Rows: 2 confidently-correct and 2
misclassified melanomas, 1 correct and 1 misclassified basal cell carcinoma,
and 1 correct nevus as a benign reference. Misclassified melanomas show the
model attending to lesion regions that resemble the (incorrect) predicted
class -- a qualitative explanation for the gap between argmax and
threshold-adjusted melanoma sensitivity (Table 4b).

In [ ]:
test_pred_df['pred_proba'] = test_pred_df[proba_cols].max(axis=1)


def pick(label, correct, n):
    if correct:
        subset = test_pred_df[(test_pred_df['label_str'] == label) & (test_pred_df['pred_str'] == label)]
        sort_col = f"proba_{label.replace(' ', '_')}"
    else:
        subset = test_pred_df[(test_pred_df['label_str'] == label) & (test_pred_df['pred_str'] != label)]
        sort_col = 'pred_proba'
    return subset.sort_values(sort_col, ascending=False).head(n)


examples = pd.concat([
    pick('melanoma', correct=True, n=2),
    pick('melanoma', correct=False, n=2),
    pick('basal cell carcinoma', correct=True, n=1),
    pick('basal cell carcinoma', correct=False, n=1),
    pick('nevus', correct=True, n=1),
]).reset_index(drop=True)

n = len(examples)
fig, axes = plt.subplots(n, 3, figsize=(9, 3.0 * n))
col_titles = ['Original', 'Grad-CAM++', 'Overlay']

for row_idx, (_, row) in enumerate(examples.iterrows()):
    path = resolve_image_path(row, cfg)
    image_pil, tensor = load_image_tensor(path)

    pred_idx = CLASSES.index(row['pred_str'])
    cam, p = gradcam.generate(tensor, pred_idx)
    overlay = overlay_heatmap(image_pil, cam)

    correct = row['label_str'] == row['pred_str']
    tag = 'correct' if correct else 'MISCLASSIFIED'
    label_text = (
        f"True: {class_label(row['label_str'])}\n"
        f"Pred: {class_label(row['pred_str'])} ({p[pred_idx]:.2f})\n"
        f"[{tag}]"
    )

    axes[row_idx, 0].imshow(image_pil)
    axes[row_idx, 0].axis('off')
    axes[row_idx, 0].text(
        -0.05, 0.5, label_text, transform=axes[row_idx, 0].transAxes,
        fontsize=9, va='center', ha='right',
    )

    axes[row_idx, 1].imshow(cam, cmap='jet')
    axes[row_idx, 1].axis('off')

    axes[row_idx, 2].imshow(overlay)
    axes[row_idx, 2].axis('off')

for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=11, fontweight='bold')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'figure5_gradcam_examples.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved figure5_gradcam_examples.png")

**Figure 6.** Mean Grad-CAM++ heatmap per class, averaged over up to 30
correctly-classified test images per class (model attention for the true
class). Consistently central, lesion-shaped activation indicates the model
relies on the lesion itself rather than borders or skin background; classes
with few correct predictions (e.g. squamous cell carcinoma) are averaged over
fewer images and may look noisier.

In [ ]:
N_PER_CLASS = 30
mean_cams = {}
n_used = {}

for cls in CLASSES:
    cls_idx = CLASSES.index(cls)
    subset = test_pred_df[(test_pred_df['label_str'] == cls) & (test_pred_df['pred_str'] == cls)]
    if len(subset) == 0:
        subset = test_pred_df[test_pred_df['label_str'] == cls]
    n_sample = min(N_PER_CLASS, len(subset))
    sample = subset.sample(n=n_sample, random_state=SEED)
    n_used[cls] = n_sample

    cams = []
    for _, row in sample.iterrows():
        path = resolve_image_path(row, cfg)
        _, tensor = load_image_tensor(path)
        cam, _ = gradcam.generate(tensor, cls_idx)
        cams.append(cam)
    mean_cams[cls] = np.mean(cams, axis=0)

fig, axes = plt.subplots(2, 4, figsize=(14, 7.5))
im = None
for i, cls in enumerate(CLASSES):
    ax = axes[i // 4, i % 4]
    im = ax.imshow(mean_cams[cls], cmap='jet', vmin=0, vmax=1)
    ax.axis('off')
    ax.set_title(f"{class_label(cls)}\n(n={n_used[cls]})", fontsize=10)

fig.colorbar(im, ax=axes, shrink=0.7, label='Mean Grad-CAM++ activation (normalized)')
fig.savefig(FIGURES_DIR / 'figure6_mean_heatmaps_per_class.png', dpi=DPI, bbox_inches='tight')
plt.show()
print("Saved figure6_mean_heatmaps_per_class.png")

## Limitations

**Overall performance is moderate, not clinical-grade.** Macro AUC is 0.873
[0.851, 0.892] and overall accuracy (argmax) is 0.675. This is a research /
graduation project, not a validated diagnostic tool.

**Melanoma sensitivity vs. overall accuracy is a real trade-off, not a solved
problem.** Under argmax, melanoma sensitivity is only 0.435 [0.374, 0.496] --
too low to be safe as a stand-alone triage signal. The per-class
Youden's-J threshold raises melanoma sensitivity to 0.851, but at the cost of
overall accuracy dropping from 0.675 to 0.488 and Cohen's kappa dropping from
0.436 to 0.285 (Table 4b). Phase 7's clinical RAG layer mitigates this by
applying a melanoma "safety-net" rule on top of the model output, but this is
a heuristic, not a validated clinical decision rule.

**Rare-class metrics are unreliable.** Squamous cell carcinoma (n=43),
dermatofibroma (n=32), vascular lesion (n=24), and actinic keratosis (n=34)
have wide bootstrap confidence intervals (Table 2). SCC sensitivity (0.140,
CI [0.044, 0.257]) in particular should not be trusted as a point estimate --
the dataset simply does not contain enough SCC test images to estimate it
precisely.

**Possible image-level leakage between splits.** The train/val/test split
(Phase 2) is a stratified random split over individual images, not grouped by
lesion/patient ID. HAM10000 contains multiple images of the same lesion taken
at different angles/zoom levels; if such near-duplicates were split across
train and test, reported test metrics may be optimistic relative to a true
patient-level holdout. A lesion-grouped re-split (e.g. `GroupShuffleSplit` on
`lesion_id`) would be needed to rule this out.

**Single train/test split -- no cross-validation.** Bootstrap CIs (Table 2)
capture sampling variance of the *test set* given this one model, not
variance across different train/val/test partitions or training runs (the
model was trained once with `SEED=42`).

**Dataset composition and demographic bias.** The dataset combines HAM10000
(predominantly Austrian/Australian dermoscopy images, lighter Fitzpatrick skin
types) with additional ISIC images for the 8th class (SCC). The model has not
been evaluated on images from darker skin tones, different dermoscopes, or
non-dermoscopic (clinical photo) images, all of which are known to degrade
skin lesion classifier performance.

**Grad-CAM++ corner artifact.** Figure 6 (mean heatmaps) shows a consistent
high-activation region in one corner across *all* eight classes. This is most
likely an artifact of EfficientNet-B4's final feature map (a small number of
spatial positions can carry high-norm, image-content-independent activations,
a phenomenon also reported in other CNN/ViT backbones) rather than a
clinically meaningful pattern. It means the per-image Grad-CAM++ overlays
(Figure 5) should be interpreted qualitatively, and the corner region should
be discounted.

**No external/multi-center validation.** All splits derive from the same two
source datasets (HAM10000, ISIC). Performance on images from a different
clinic, camera, or population is unknown.

## Ethical Statement

**Not a medical device.** This project is an academic/graduation exercise. It
has not been reviewed, validated, or approved by any regulatory or clinical
body, and **must not be used to make or influence real diagnostic or
treatment decisions**. The Gradio interface (next section) displays this
disclaimer prominently and the Phase 7 clinical RAG report always recommends
professional medical evaluation.

**Risk of false reassurance.** With argmax melanoma sensitivity of only
43.5%, more than half of melanomas in this test set would be missed if the
raw model output were taken at face value. This is precisely the failure mode
that is most dangerous in a screening context (a missed melanoma vs. a
false-positive benign lesion flagged for review). The threshold-adjusted
decision rule and the Phase 7 melanoma safety-net rule are both attempts to
shift errors away from false negatives for malignant classes, at a known cost
to overall accuracy -- this trade-off is made explicit, not hidden.

**Data provenance and consent.** HAM10000 and the supplementary ISIC images
are public, de-identified dermoscopy datasets distributed for research use
under their respective licenses (CC-BY-NC for HAM10000; ISIC images are
released under CC-0 / CC-BY depending on contributing center). No new patient
data was collected. No attempt is made to re-identify patients.

**Demographic fairness.** As noted in Limitations, the underlying datasets
are not demographically representative of the global population (skin tone,
age, geography). A model trained on this data could perform worse for
under-represented groups. Any future deployment beyond a research demo would
need a fairness audit across Fitzpatrick skin types before being presented to
real users.

**Intended use.** The Gradio demo is intended as (a) a way to showcase the
trained model's predictions and explanations (Grad-CAM++) for educational
purposes, and (b) a research artifact demonstrating an end-to-end pipeline
(classification + explainability + a retrieval-augmented clinical-context
report). It is explicitly **not** intended for patients or clinicians to use
for real triage decisions.

## Reproducibility

A full reproducibility package is provided in
[`reproduce.md`](reproduce.md) and [`requirements.txt`](requirements.txt) at
the project root. In summary:

- **Environment**: Python 3.11, PyTorch 2.5.1+cu121, `SEED=42` fixed for
  `numpy`, `torch`, and all `sklearn` splits/bootstraps.
- **Pipeline**: Phases 1-2 (EDA + preprocessing/splits) produce
  `data/{train,val,test}.csv` and `data/preprocessing_config.json`. Phase 3-4
  train the EfficientNet-B4 + SE-block model in 3 progressive-unfreezing
  stages, producing `models/checkpoints/final_model.pth` and
  `results/metrics/{training_summary,history_stage*}.csv`. Phase 5 evaluates
  on the held-out test set, producing `results/metrics/test_*.csv`. Phase 6
  generates Grad-CAM++ visualizations. Phase 7 builds the clinical RAG report
  generator. Phase 8 (this notebook) compiles all of the above into the
  tables and figures saved under `results/tables/` and `results/figures/`.
- **This notebook only reads existing artifacts** -- no retraining is
  performed here, so it can be re-run quickly (a few minutes, dominated by
  the Grad-CAM++ figures) as long as `final_model.pth` and the `results/metrics/*.csv`
  files from Phases 1-7 are present.
- **Raw image data** (HAM10000, ISIC) is not included in this repository due
  to size and license terms; `reproduce.md` links to the original sources.